# Example sheet for robot arm tasks

This work sheet will take you through how the robot gym environment works, and how you can generate a task, get information about the environment and act on it. 

Firstly we will install the libraries needed which is made of mujoco (a physics simulator developed by deepmind) and dm_control which controls the inverse kinematics (controlling the robotic arm) as well as a few other useful libraries. This may take a bit of time dependinding on your computer and internet connection.

In [ ]:
!python -m pip install --upgrade pip setuptools wheel
!pip install numpy 
!pip install mujoco
!pip install dm_control
!pip install matplotlib

  Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Using cached packaging-26.1-py3-none-any.whl (95 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 63.2.0
    Uninstalling setuptools-63.2.0:
      Successfully uninstalled setuptools-63.2.0
  Attempting uninstall: pip
    Found existing installation: pip 23.1.2
    Uninstalling pip-23.1.2:
      Successfully uninstalled pip-23.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 23.1
    Uninstalling packaging-23.1:
      Successfully uninstalled packaging-23.1
  Using cached mujoco-3.7.0-cp310-cp310-win_amd64.whl.metadata (43 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached etils-1.13.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached glfw-2.10.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38.py39.py310.py311.py312.py313.py314-none-win_amd64.whl.metadata (5.4 kB)
  Us

In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import sys
sys.path.append("../../..")
from Simulation.Mujoco.environment import *
#from Simulation.PyBullet.tasks import *
#from Controller.languageModel import *

## Booting the simulation 
Do this once to boot the simulator. Once it is booted you do not need to call Env again. If you want to put it back to original settings, you can call env.reset()

replace the line ```C:\Users\dexte\Documents\GitHub\Robot_shape_learning\Assets\kuka_iiwa_141``` with the path to your directory

In [2]:
e=Env("C:/Users/dexte/Documents/GitHub/Robot_shape_learning/Assets/kuka_iiwa_14/",realtime=1,speed=1/240)
e.reset()
viewer= mj.viewer.launch_passive(e.model, e.data)


### Moving the arm
The arm uses inverse kinematics to move to specific points in the environment. Have a go at changing the coordinates to move the arm around. The units of the coordinates are in meters.

What are the constraints? What happens if you go out of the bounds? Understanding the limitations of robotics is important. 

Your current task is to change the coordinates in ```target__coord``` to different values, for example [0.4,0.3,0.5]. Look at where the arm moves to get an idea of how you can control it. 

In [ ]:
target_coord=[0.4,-0.3,0.2] #TASK: change the values
e.move_gripper_to(target_coord) #actually move the robot
e.step(100,viewer=viewer) #update simulation, required at the end of each action


Lets add an object to the environment,we can pick up an object by hovering over it with ```move_gripper_to``` and calling a function ```pick_block```

In [5]:
e.reset()
#generate the block in the simulation
e.generate_blocks(1) #generate 1 block in a random place
cube_pos, _ = e.getBasePositionAndOrientation(e.block_ids[0]) #get information about this block
cube_pos=list(cube_pos)
cube_pos[2]+=0.08 #increase the z position so the robot hovers over it and does not crush it
e.move_gripper_to(cube_pos)
e.step(100,viewer=viewer)

e.pick_block(e.block_ids[0]) #pick it up
up=np.array(cube_pos)
up[2]+=0.6 #increase height to show it is attached
e.move_gripper_to(up)
e.step(100,viewer=viewer) #update sim

KeyError: "Invalid name 'block_0'. Valid names: ['base', 'link1', 'link2', 'link3', 'link4', 'link5', 'link6', 'link7', 'world']"

You can drop the block by calling ```put_block() ```

In [5]:
env.put_block()
env.step(100) #update sim

Finally we can get details about the environment by calling the observation function. It returns a dictionary of items

In [6]:
dictionary=env.get_observation()
print("Block coordinates:",dictionary['blocks'])
print("Block colours:",dictionary['block_colours'])
print("Block sizes:",dictionary['sizes'])

print("All data items",list(dictionary.keys()))

Block coordinates: [(0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754)]
Block colours: [(0.5671306516623507, 0.3247738750760214, 0.607158150622296, 1.0)]
Block sizes: [1]
All data items ['blocks', 'block_colours', 'robot_end_position', 'holding_constraint', 'block_name', 'sizes', 'contacts', 'holding']


Play around with the functions, try generate more blocks and select different ones. Here are a few challenges to help you get used to the robot control

In [7]:
# Make the robot pick up block and move it to a different location

In [8]:
# Make the robot generate two blocks and place one on top of the other

## Hard code challenges
Here are a few preset challenges. Can you hard code a solution that works? You can make use of the get observation function to look at objects and information such as colours and sizes, then use the coordinates to move to each. 

### Task 1 - Arrange into a tower
![My GIF](https://raw.githubusercontent.com/shepai/Robot_shape_learning/refs/heads/main/Assets/Gifs/task1_fast.gif)

In [ ]:
#Task 1: arrange into a tower
task=task1()
task.generate(env) #make the blocks show

### Task 2 - sort the colours into three seperate towers

![My GIF](https://raw.githubusercontent.com/shepai/Robot_shape_learning/refs/heads/main/Assets/Gifs/task2_fast.gif)

In [ ]:
# Task 2
task=task2()
task.generate(env) #make the blocks show

## Using AI
We could use reinforcment learning or genetic algorithms to optimize neural controllers. This would take a large amount of time. Can we make use of existing large language models for robotic control?

You will need to make sure to install the ollama library from 
https://ollama.com/download/windows

or for linux
```[bash]
pip install ollama
curl -fsSL https://ollama.com/install.sh | sh
ollama pull gemma3
ollama pull llama3
```
You can chose either model as long as you use the one downloaded

```[bash]
ollama pull gemma3
ollama pull llama3
```
Have a look at the prompt we are using for control

In [11]:
decision = Decisions(model="gemma3")

the_prompts = decision.system_prompt
print(the_prompts)



['You are a robot arm which can pick up objects with its tractor beam on the hand. You just need to hover the gripper over the z with an offset of 0.15 where the coordinates are in the form (x,y,z) otherwise you crush the object', 'Your only actions you can do are move_gripper_to(target_coord) to move the gripper hand to a specific coordinate, pick_block(index) to pick up a specific block, put_block() to drop whatever is held. Given a task generate the steps to solving it using only these listed commands', ' ']


We will also need to be able to convert our observations of the environment into a text prompt. We have written the code for this for you. Feel free to play around if you can think of a better way to do this?

In [ ]:
def observations_to_text(obs):
    string="Environment information"
    for key in obs:
        if type(obs[key])==type([]):
            string+=key+": "
            for i in range(len(obs[key])):
                try:
                    string+=str(np.round(np.array(obs[key][i]),2))+", "
                except:
                    string+=str(obs[key][i])+", "
        else:
            string+=key+": "+str(obs[key])
        string+="\n"
    return string
print(observations_to_text(env.get_observation()))

Environment informationblocks: (0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754), 
block_colours: (0.5671306516623507, 0.3247738750760214, 0.607158150622296, 1.0), 
robot_end_position: (0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335)
holding_constraint: None
block_name: cube_small.urdf, 
sizes: 1, 
contacts: [], 
holding: None



In [ ]:
import re

def segment_function_calls(input_string):
    # Define a regex pattern to capture function names and parameters
    pattern = r'(\w+)\((.*?)\)'  # This matches the function name followed by parameters inside parentheses
    matches = re.findall(pattern, input_string)
    
    # Organize the matches into function names and their parameters
    function_calls = []
    for match in matches:
        function_name = match[0]
        parameters_str = match[1].strip()
        
        # Handle cases for both [.., .., ..] and .., .., ..
        if parameters_str.startswith('[') and parameters_str.endswith(']'):
            parameters = parameters_str[1:-1].split(',') if parameters_str[1:-1] else []
        else:
            parameters = parameters_str.split(',') if parameters_str else []
        
        # Clean any leading/trailing whitespace for each parameter
        parameters = [param.strip() for param in parameters]
        
        function_calls.append({'function': function_name, 'parameters': parameters})
    
    return function_calls

# Example usage:
input_string = reply

result = segment_function_calls(input_string)

for call in result:
    print(f"Function: {call['function']}, Parameters: {call['parameters']}")
    if "pick_block" in call['function']:
        env.pick_block(int(call['parameters'][0].replace("index=","")))
    elif "move_gripper_to" in call['function']:
        coords=call['parameters']
        coords[0]=float(coords[0].replace("x=",""))
        coords[1]=float(coords[1].replace("y=",""))
        coords[2]=float(coords[2].replace("z=",""))
        env.move_gripper_to(coords)
    elif "put_block" in call['function']:
        env.put_block()
    env.step(100)

    

Function: pick_block, Parameters: ['index=0']
Already holding a block!
Function: move_gripper_to, Parameters: ['x=0.5', 'y=0.0', 'z=0.05']
Function: put_block, Parameters: []
Function: pick_block, Parameters: ['index=1']
Function: move_gripper_to, Parameters: ['x=0.5', 'y=0.1', 'z=0.05']
Function: put_block, Parameters: []


The prompt is automattically used, but you can change it by changing the_prompts [0] like so
```[python]
the_prompt[0]="new prompt"
```

We set the task and generate action using the following:

In [ ]:
env.reset()
env.generate_blocks(2) #generate 2 block in a random place
decision.set_task("You are given two blocks and you need to stack them on top of eachother")
reply=decision.chat(observations_to_text(env.get_observation()))
print(reply)

Okay, I understand. I have the following information:

*   **Blocks:** I have a block with coordinates (0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754).
*   **Block Colors:** The block is red (approximately).
*   **Robot End Position:** The robot's end position is (0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335).
*   **Holding Constraint:** None.
*   **Block Name:** `cube_small.urdf`
*   **Size:** 1
*   **Contacts:** None.
*   **Holding:** None

I will now proceed to pick up the block and place it on top of the other block.

```
pick_block(0)
move_gripper_to(0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335)
put_block()
```


### Task - write code that locates the functions it is outputting in order, and get the robot to run through them

In [ ]:
import re

def segment_function_calls(input_string):
    # Define a regex pattern to capture function names and parameters
    pattern = r'(\w+)\((.*?)\)'  # This matches the function name followed by parameters inside parentheses
    matches = re.findall(pattern, input_string)
    
    # Organize the matches into function names and their parameters
    function_calls = []
    for match in matches:
        function_name = match[0]
        parameters_str = match[1].strip()
        
        # Handle cases for both [.., .., ..] and .., .., ..
        if parameters_str.startswith('[') and parameters_str.endswith(']'):
            parameters = parameters_str[1:-1].split(',') if parameters_str[1:-1] else []
        else:
            parameters = parameters_str.split(',') if parameters_str else []
        
        # Clean any leading/trailing whitespace for each parameter
        parameters = [param.strip() for param in parameters]
        
        function_calls.append({'function': function_name, 'parameters': parameters})
    
    return function_calls

# Example usage:
input_string = reply

result = segment_function_calls(input_string)

for call in result:
    print(f"Function: {call['function']}, Parameters: {call['parameters']}")
    if "pick_block" in call['function']:
        # code here
    elif "move_gripper_to" in call['function']:
        # code here
    elif "put_block" in call['function']:
        # code here
    env.step(100)

    